# Gate settings sweep (Plan 01): driver-top3

Runs `experiments/sweep.py` with `experiments/sweeps/gates_driver-top3.yaml`. That is 36 gate settings (gate learning rate × gate start θ × λ) plus Base, each with 3 seeds, so **111 training runs**. It reports the mean validation ROC-AUC of every setting.

The notebook works on any device. It uses a GPU when one is available and the CPU otherwise.

**On Colab:**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. *Runtime → Run all*.
   - The setup cell installs the pinned environment (about 5 minutes).
   - The quick check runs 2 short runs.
   - Then the full sweep runs.
3. If Colab disconnects, run all cells again. With `USE_DRIVE = True`, results are kept in Google Drive, and runs that already finished are skipped.
4. At the end, the last cell downloads a zip of the results folder. Upload the zip (or this notebook with its outputs) so the results can be committed and analysed.

**Everything is logged.** For each run, the results folder keeps:
- one row in `results.csv`;
- a per-epoch log;
- the state of every gate.

The folder also has `env.json` (versions, GPU, git commits) and `summary.csv`. The cell outputs below record the same information.

In [ ]:
# ---- Settings ----
PROJECT_REPO = 'https://github.com/HARSHALK2598/cs224w-structural-optimizer'
PROJECT_BRANCH = 'step/01-4-sweep'               # train.py, sweep.py, the sweep config
PYG_REPO = 'https://github.com/HARSHALK2598/pytorch_geometric'
PYG_BRANCH = 'step/01-2-relational-filter'       # HardConcreteGate, RelationalFilter
RELBENCH_COMMIT = '91c1f6420a03feb61c279aaeb194d7268435f278'  # as in scripts/setup_env.sh

SWEEP_CONFIG = 'experiments/sweeps/gates_driver-top3.yaml'
RUN_QUICK_CHECK = True   # 2 runs x 2 epochs into <out>/check, before the sweep
RUN_SWEEP = True         # the full sweep (resumable)
MAX_RUNS = None          # e.g. 10 to run only part of the sweep now
USE_DRIVE = True         # Colab only: keep results in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/cs224w-structural-optimizer'

## 1. Environment

- **On Colab:** clones both repositories and installs pinned versions:
  - torch 2.12.0, CUDA 12.6 build on a GPU, CPU build otherwise;
  - pyg-lib 0.9.0;
  - the PyG fork, editable;
  - RelBench at the pinned commit;
  - PyTorch Frame 0.3.0;
  - sentence-transformers 6.2.0.

  Colab's own torchvision and torchaudio are removed, because they would not match this torch.
- **Elsewhere:** uses the current environment and repository, i.e. the one set up by `scripts/setup_env.sh`.

The training itself runs in a separate Python process (`python experiments/sweep.py`), so the fresh installs are used without restarting the notebook.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def sh(cmd, cwd=None):
    """Runs a shell command, streaming its output; stops on failure."""
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')


HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'], capture_output=True).returncode == 0

if IN_COLAB:
    ROOT = Path('/content/cs224w-structural-optimizer')
    if not ROOT.exists():
        sh(f'git clone -q --branch {PROJECT_BRANCH} {PROJECT_REPO} {ROOT}')
    else:
        sh(f'git -C {ROOT} pull -q origin {PROJECT_BRANCH}')
    pyg = ROOT / 'pytorch_geometric'
    if not pyg.exists():
        sh(f'git clone -q --branch {PYG_BRANCH} {PYG_REPO} {pyg}')
    else:
        sh(f'git -C {pyg} pull -q origin {PYG_BRANCH}')

    build = 'cu126' if HAS_GPU else 'cpu'
    pip = f'{sys.executable} -m pip install -q'
    sh(f'{sys.executable} -m pip uninstall -q -y torchvision torchaudio')
    sh(f'{pip} torch==2.12.0 --index-url https://download.pytorch.org/whl/{build}')
    sh(f'{pip} pyg_lib==0.9.0 -f https://data.pyg.org/whl/torch-2.12.0+{build}.html')
    sh(f'{pip} -e {pyg}')
    sh(f'{pip} "relbench @ git+https://github.com/snap-stanford/relbench@{RELBENCH_COMMIT}" '
       f'pytorch-frame==0.3.0 sentence-transformers==6.2.0 pyyaml pandas')
else:
    # Run from anywhere inside the project repository (or set PROJECT_ROOT).
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).resolve()
    while not (ROOT / 'experiments' / 'sweep.py').exists():
        if ROOT.parent == ROOT:
            raise RuntimeError('Start the notebook inside the project repository.')
        ROOT = ROOT.parent

os.chdir(ROOT)
print('project root:', ROOT, '| Colab:', IN_COLAB, '| GPU:', HAS_GPU)
sh(f'{sys.executable} -c "import torch, torch_geometric, torch_frame, relbench, pyg_lib; '
   f'print(\'torch\', torch.__version__, \'| cuda\', torch.cuda.is_available(), '
   f'torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'\', '
   f'\'| pyg\', torch_geometric.__version__, \'| frame\', torch_frame.__version__, '
   f'\'| relbench\', relbench.__version__, \'| pyg_lib\', pyg_lib.__version__)"')
sh('git log --oneline -1')
sh('git -C pytorch_geometric log --oneline -1' if (ROOT / 'pytorch_geometric').exists() else 'true')

## 2. Where results go

Results go to `experiments/results/sweeps/<name>/` in the repository. On Colab with `USE_DRIVE = True`, they go to the same path under `DRIVE_DIR` in your Google Drive, so they survive a disconnect.

In [ ]:
import yaml

with open(ROOT / SWEEP_CONFIG) as f:
    CFG = yaml.safe_load(f)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path(DRIVE_DIR) / 'sweeps' / CFG['name']
else:
    OUT = ROOT / 'experiments' / 'results' / 'sweeps' / CFG['name']
OUT.mkdir(parents=True, exist_ok=True)
print('sweep:', CFG['name'], '| results folder:', OUT)
print(yaml.safe_dump(CFG, sort_keys=False))

## 3. Quick check: 2 runs × 2 epochs

This takes about a minute on a GPU, plus the first-time download of `rel-f1` and the GloVe model, and building the text features (a few minutes). It goes to a separate `check/` folder, so it never mixes with the sweep's results.

In [ ]:
if RUN_QUICK_CHECK:
    check = OUT / 'check'
    shutil.rmtree(check, ignore_errors=True)
    sh(f'{sys.executable} experiments/sweep.py {SWEEP_CONFIG} --max-runs 2 --epochs 2 --out "{check}"')
    print(open(check / 'env.json').read())

## 4. The sweep

There is one line per finished run: validation and test ROC-AUC, best epoch, open gates and time. Rerunning this cell continues where the sweep stopped.

In [ ]:
if RUN_SWEEP:
    extra = f' --max-runs {MAX_RUNS}' if MAX_RUNS else ''
    sh(f'{sys.executable} experiments/sweep.py {SWEEP_CONFIG} --out "{OUT}"{extra}')

## 5. Results

- Every setting, best mean validation first. Settings are chosen on **validation**; test is for reference.
- Then validation by gate start θ (rows) and gate learning rate (columns), one table per λ.
- Then Base on this machine, for comparison.

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)

res = pd.read_csv(OUT / 'results.csv')
print(f'{len(res)} runs finished on', res['device'].iloc[0])
summary = pd.read_csv(OUT / 'summary.csv') if (OUT / 'summary.csv').exists() else None
print(summary.round(4).to_string(index=False) if summary is not None else 'no summary yet')

flt = res[res['model'] == 'filter']
if len(flt):
    for lam, part in flt.groupby('lambda_col'):
        table = part.pivot_table(index='gate_init', columns='gate_lr',
                                 values='val_roc_auc', aggfunc='mean').round(4)
        print(f'\nmean validation ROC-AUC, lambda = {lam:g}  (rows: gate start, columns: gate lr)')
        print(table.to_string())
base = res[res['model'] == 'base']
if len(base):
    print(f"\nBase on this machine: val {base['val_roc_auc'].mean():.4f} ± "
          f"{base['val_roc_auc'].std():.4f}, test {base['test_roc_auc'].mean():.4f} ± "
          f"{base['test_roc_auc'].std():.4f} ({len(base)} seeds)")
failures = OUT / 'failures.log'
print('\nfailures:', failures.read_text() if failures.exists() else 'none')

## 6. Download the results

This writes `<name>.zip` with everything in the results folder: `results.csv`, `summary.csv`, `env.json`, `logs/` and `gates/`. On Colab it downloads it. Upload the zip back to the project, and save this notebook with its outputs (*File → Download → .ipynb*).

In [ ]:
import tempfile
zip_dir = Path('/content') if IN_COLAB else Path(tempfile.gettempdir())
archive = shutil.make_archive(str(zip_dir / CFG['name']), 'zip', root_dir=OUT)
print('wrote', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)